In [1]:
import pandas as pd
import joblib
from pathlib import Path

In [30]:
#Path setup
base_dir = Path("..")
data_dir = base_dir / "data" / "processed"
model_dir = base_dir / "models"#/ "no_model_modis"
preprocessing_dir = base_dir / "data" / "processed" / "preprocessing" / "no_modis"


# load data satelit
#df_predict = pd.read_csv(data_dir / "jakarta_tanpa_Modis_1km_2023.csv") #done
#df_predict = pd.read_csv(data_dir / "jakarta_tanpa_Modis_1km_2024.csv") #done
df_predict = pd.read_csv(data_dir / "jakarta_tanpa_Modis_1km_2025.csv")

print(df_predict.shape)
df_predict.head()

(658, 7)


,lat,lon,s5p_co,s5p_no2,s5p_o3,s5p_so2,viirs_ntl
0,-6.364564,106.886044,0.032389,0.000071,0.116598,0.000089,30.452932
1,-6.364564,106.895027,0.032423,0.000069,0.116594,0.000032,21.725202
2,-6.364564,106.912994,0.032622,0.000069,0.116500,0.000064,25.892045
3,-6.355581,106.796213,0.032594,0.000072,0.116421,0.000033,31.800545
4,-6.355581,106.805196,0.032570,0.000079,0.116511,0.000046,34.711155


<h2>1.1 Pisahkan Data</h2>

In [31]:
df_metadata = df_predict[["lat", "lon"]].copy()

df_metadata

,lat,lon
0,-6.364564,106.886044
1,-6.364564,106.895027
2,-6.364564,106.912994
3,-6.355581,106.796213
4,-6.355581,106.805196
...,...,...
653,-6.095069,106.742314
654,-6.095069,106.751297
655,-6.086086,106.733330
656,-6.086086,106.751297


<h4>1.2 Rename kolom samakan</h4>

In [32]:
rename_map = {
    "viirs_ntl": "viirs_VIIRS_NTL"
}

df_predict = df_predict.rename(columns=rename_map)
print("Kolom setelah rename:", df_predict.columns.tolist())

Kolom setelah rename: ['lat', 'lon', 's5p_co', 's5p_no2', 's5p_o3', 's5p_so2', 'viirs_VIIRS_NTL']


In [33]:
feature_cols = [
    "s5p_co",
    "s5p_no2",
    "s5p_o3",
    "s5p_so2",
    "viirs_VIIRS_NTL"
]
X_predict = df_predict[feature_cols].copy()
X_predict.head()

,s5p_co,s5p_no2,s5p_o3,s5p_so2,viirs_VIIRS_NTL
0,0.032389,0.000071,0.116598,0.000089,30.452932
1,0.032423,0.000069,0.116594,0.000032,21.725202
2,0.032622,0.000069,0.116500,0.000064,25.892045
3,0.032594,0.000072,0.116421,0.000033,31.800545
4,0.032570,0.000079,0.116511,0.000046,34.711155


In [34]:
# Load trained objects
knn_imputer = joblib.load(preprocessing_dir / "knn_imputer_no_modis.pkl")
robust_scaler = joblib.load(preprocessing_dir / "robust_scaler_no_modis.pkl")
xgb_model = joblib.load(model_dir / "xgb_model_test_no_modis.pkl")

<h2>2. Transform Data</h2>

In [35]:
#KNN Imputation
X_imputed = knn_imputer.transform(X_predict)

# Scaler/Rovust Scaling
X_scaled = robust_scaler.transform(X_imputed)

C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


<h4>2.1 Pred Kelas Kualitas Udara</h4>

In [36]:
y_pred = xgb_model.predict(X_scaled)
#y_proba = xgb_model.predict_proba(X_scaled)


<h2>3. Gabung prediksi dengan data</h2>

In [37]:
df_result = df_metadata.copy()
df_result["predicted_class"] = y_pred

In [38]:
out_path = data_dir / "prediction_results_no_modis_2025.csv"

df_result.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: ..\data\processed\prediction_results_no_modis_2025.csv


In [39]:
class_map = {
    0 : "GOOD",
    1 : "MEDIUM",
    2 : "UNHEALTHY"
}

df_result["predicted_label"] = df_result["predicted_class"].map(class_map)

#check
print(df_result[["predicted_class", "predicted_label"]].head())

#save output
#out_path = data_dir / "predicted_label_no_modis_2023.csv" #done
#out_path = data_dir / "predicted_label_no_modis_2024.csv" #done
out_path = data_dir / "predicted_label_no_modis_2025.csv"

df_result.to_csv(out_path, index=False)

print("Saved to:", out_path)

   predicted_class predicted_label
0                2       UNHEALTHY
1                1          MEDIUM
2                2       UNHEALTHY
3                2       UNHEALTHY
4                1          MEDIUM
Saved to: ..\data\processed\predicted_label_no_modis_2025.csv


In [40]:
df_result["predicted_label"].value_counts()


predicted_label
MEDIUM       427
UNHEALTHY    130
GOOD         101
Name: count, dtype: int64